In [ ]:
import os
import numpy as np
import torch
import csv 

from src.resnet import ResNet18
from src.inference import extract_features
from src.pca import PCATransformer
from src.draw_figures import *
from src.dataloader import build_excluded_dataset, get_dataset_config
from src.paths import DATA_DIR, FIGURES_DIR, MODELS_DIR, OUTPUT_DIR, ensure_dir
from src.umap_transformer import UMAPTransformer

from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from sklearn.metrics import f1_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


%matplotlib inline

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
ensure_dir(DATA_DIR) 
ensure_dir(MODELS_DIR) 
ensure_dir(FIGURES_DIR) 

# filename = "./output.csv"
# with open(filename, mode='w', newline='') as file:
#     writer = csv.writer(file)
    
#     writer.writerow([
#         'target', 
#         'model_f1_score',
#         'lda_pca_auc',
#         'lda_umap_auc', 
#         'lda_comb_auc',
#         'new_pipeline_f1_pca',
#         'new_pipeline_f1_umap',
#         'new_pipeline_f1_comb',
#         'model_f1_score_list',
#         'new_pipeline_f1_list_pca',
#         'new_pipeline_f1_list_umap',
#         'new_pipeline_f1_list_comb',
#     ])




### 1. Подготовка данных:

In [ ]:
dataset_name = "cifar10"
dataset_config = get_dataset_config(dataset_name)
all_class_ids = list(range(dataset_config.num_classes))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(dataset_config.mean, dataset_config.std)
])

target = 0
generator = torch.Generator().manual_seed(42)
output_dir = ensure_dir(OUTPUT_DIR / dataset_name / str(target)) 

### То, на чем обучаем resnet, строим pca
train_dataset = build_excluded_dataset(
    name=dataset_name,
    root=str(DATA_DIR),
    exclude_class=[target],
    train=True, 
    transform=transform
)

train_size = int(0.8 * len(train_dataset))
val_size = int(0.15 * len(train_dataset))

# выборка для построение PCA пространства, для тренеровки модели разделения
test_size = (len(train_dataset) - train_size - val_size) // 2 
# выбока, которая отрпаится в финальный тест нового пайплайна
new_pipeline_test_size = len(train_dataset) - train_size - val_size - test_size


train, val, test, new_pipeline_test = random_split(
    train_dataset, 
    [train_size, val_size, test_size, new_pipeline_test_size], 
    generator=generator
)

train_loader = DataLoader(train, batch_size=64, shuffle=True, num_workers=2, generator=generator)
val_loader   = DataLoader(val, batch_size=64, shuffle=False, num_workers=2, generator=generator)
test_loader = DataLoader(test, batch_size=64, shuffle=False, num_workers=2, generator=generator)

### То, на чем делаем холостой (все классы будует неверные) предикт ради feature layer, на чем обучаем и проводим тесты lda
target_dataset = build_excluded_dataset(
    name=dataset_name,
    root=str(DATA_DIR),
    exclude_class=[class_id for class_id in all_class_ids if class_id != target],
    train=True, 
    transform=transform
)

target_train_size = int(0.85 * len(target_dataset))
target_test_size = (len(target_dataset) - target_train_size)
other = len(target_dataset) - target_train_size - target_test_size

train_target, test_target, orher = random_split(
    target_dataset, 
    [target_train_size, target_test_size, other]
)

train_target_loader = DataLoader(train_target, batch_size=64, shuffle=True, num_workers=2, generator=generator)
test_target_loader = DataLoader(test_target, batch_size=64, shuffle=True, num_workers=2, generator=generator)

### То, на чем пробуем новый пайплайн
new_pipeline_loader = DataLoader(new_pipeline_test+test_target, batch_size=64, shuffle=True, num_workers=2, generator=generator)


In [ ]:
len(train_dataset), len(target_dataset), len(train), len(val), len(test), len(train_target), len(test_target)

### 2. Обучение и inference модели на train наборе данных

In [ ]:
resnet = ResNet18(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        learning_rate=0.001,
        num_classes=dataset_config.num_classes,
        input_channels=dataset_config.input_channels,
        num_epochs=8,
        # weights=MODELS_DIR / f"resnet18_{dataset_name}_without{target}.pth"
    )

history = resnet.train()

In [ ]:
plot_loss(history['train_loss'], history['val_loss'], path=str(output_dir / "train_loss.png"))

In [ ]:
plot_accuracy(history['val_accuracy'], path=str(output_dir / "accuracy.png"))

In [ ]:
plot_accuracy(history['val_f1_macro'], "f1_score", path=str(output_dir / "f1_score.png"))

In [ ]:
resnet.save_model(MODELS_DIR / f"resnet18_{dataset_name}_without{target}.pth")

### 3. Извлечение признаков модели

In [ ]:
test_cifar9_features, test_cifar9_preds, test_cifar9_gt_preds = extract_features(resnet.model, test_loader, device)
test_cifar9_iscorrect = test_cifar9_gt_preds == test_cifar9_preds

mask = test_cifar9_gt_preds == test_cifar9_preds
test_cifar9_CR = test_cifar9_features[mask == True]
test_cifar9_WR = test_cifar9_features[mask == False]

test_cifar9_f1_score =  f1_score(test_cifar9_gt_preds, test_cifar9_preds, average="macro")
test_cifar9_f1_score_list = f1_score(test_cifar9_gt_preds, test_cifar9_preds, average=None)

print("f1-мера (глобальные TP и FP): {}, \nПо каждому классу: {}".format(
        test_cifar9_f1_score,
        test_cifar9_f1_score_list
    )
)

In [ ]:
train_target_features, train_target_preds, train_target_gt_preds = extract_features(resnet.model, train_target_loader, device)

mask = train_target_gt_preds == train_target_preds
train_target_CR = train_target_features[mask == True]  
train_target_WR = train_target_features[mask == False] 

len(train_target_CR), len(train_target_WR)

#### 3.1 Построение и процецирование в pca пространство:

In [ ]:
pca = PCATransformer(whiten=True) 
pca.fit(test_cifar9_features)


test_cifar9_CR_p = pca.transform(test_cifar9_CR)
test_cifar9_WR_p = pca.transform(test_cifar9_WR)

                            
# plot_projection(test_cifar9_CR_p, test_cifar9_WR_p, path=str(output_dir / "pca_proj.png"))
plot_projection(test_cifar9_CR_p, test_cifar9_WR_p)

In [ ]:
test_cifar9_features_p = pca.transform(test_cifar9_features)

print("Проецирование всех test данных:")
plot_embedding_3d(test_cifar9_features_p, test_cifar9_gt_preds.astype(str))

##### Target выборка

In [ ]:
train_target_p = pca.transform(train_target_features)
# plot_projection(test_cifar9_CR_p, test_cifar9_WR_p, target_data=train_target_p, path=str(output_dir / "pca_proj_with_target.png"))
plot_projection(test_cifar9_CR_p, test_cifar9_WR_p, target_data=train_target_p)

In [ ]:
test_cifar9_features_p_with_target_p = np.vstack((test_cifar9_features_p, train_target_p))
test_cifar9_gt_preds_with_target = np.concatenate((test_cifar9_gt_preds, train_target_gt_preds))

print(f"Проецирование всех test данных c target классом ({target}):")
plot_embedding_3d(test_cifar9_features_p_with_target_p, test_cifar9_gt_preds_with_target.astype(str))

#### 3.2 Построение и процецирование в umap пространство:

In [ ]:
umap = UMAPTransformer(n_components=3, n_neighbors=5, min_dist=0.1, metric='cosine')
umap.fit(test_cifar9_features)

test_cifar9_CR_u = umap.transform(test_cifar9_CR)
test_cifar9_WR_u = umap.transform(test_cifar9_WR)

In [ ]:
plot_projection(test_cifar9_CR_u, test_cifar9_WR_u, method="umap", path=str(output_dir / "umap_proj.png"))

In [ ]:
test_cifar9_features_u = umap.transform(test_cifar9_features)
plot_embedding_3d(test_cifar9_features_u, test_cifar9_gt_preds)

##### Target выборка

In [ ]:
train_target_u = umap.transform(train_target_features)
plot_projection(test_cifar9_CR_u, test_cifar9_WR_u, target_data=train_target_u, method="umap", path=str(output_dir / "umap_proj_with_target.png"))

In [ ]:
test_cifar9_features_u_with_target_u = np.vstack((test_cifar9_features_u, train_target_u))


print(f"Проецирование всех test данных c target классом ({target}):")
plot_embedding_3d(test_cifar9_features_u_with_target_u, test_cifar9_gt_preds_with_target.astype(str))

#### 3.3 Комбинированный методом

In [ ]:
umap_comb = UMAPTransformer(n_components=3, n_neighbors=15, min_dist=0.1, metric='cosine')
umap_comb.fit(test_cifar9_features_p)

test_cifar9_CR_pu = umap_comb.transform(test_cifar9_CR_p)
test_cifar9_WR_pu = umap_comb.transform(test_cifar9_WR_p)

In [ ]:
plot_projection(test_cifar9_CR_pu, test_cifar9_WR_pu, method="umap")

In [ ]:
test_cifar9_features_pu = umap_comb.transform(test_cifar9_features_p)
plot_embedding_3d(test_cifar9_features_pu, test_cifar9_gt_preds)

##### Target выборка:

In [ ]:
train_target_pu = umap_comb.transform(train_target_p)
plot_projection(test_cifar9_CR_pu, test_cifar9_WR_pu, target_data=train_target_pu)

In [ ]:
test_cifar9_features_pu_with_target_pu = np.vstack((test_cifar9_features_pu, train_target_pu))


print(f"Проецирование всех test данных c target классом ({target}):")
plot_embedding_3d(test_cifar9_features_pu_with_target_pu, test_cifar9_gt_preds_with_target.astype(str))

### 4. (Исключительно вариант курсовой) Обучние lda target vs cr + wr

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

#### 4.1 Для PCA:

In [ ]:
X_pca = np.vstack((test_cifar9_CR_p, test_cifar9_WR_p, train_target_p))

# 0 = старые классы (CR+WR), 1 = новый unseen класс
y_pca = np.hstack((
    np.zeros(len(test_cifar9_CR_p) + len(test_cifar9_WR_p)),
    np.ones(len(train_target_p))
))

lda_on_pca = LinearDiscriminantAnalysis(n_components=1)
X_pca_lda = lda_on_pca.fit_transform(X_pca, y_pca)

In [ ]:
scores = X_pca_lda.ravel()  

lda_pca_auc = plot_roc_auc(scores, y_pca, path=str(output_dir / "pca_roc_auc_for_lda.png"))

In [ ]:
plot_kde(X_pca_lda, y_pca, path=str(output_dir / "pca_kde_for_lda.png"))

#### 4.2 Для UMAP:

In [ ]:
X_umap = np.vstack((test_cifar9_CR_u, test_cifar9_WR_u, train_target_u))

# 0 = старые классы (CR+WR), 1 = новый unseen класс
y_umap = np.hstack((
    np.zeros(len(test_cifar9_CR_u) + len(test_cifar9_WR_u)),
    np.ones(len(train_target_u))
))

lda_on_umap = LinearDiscriminantAnalysis(n_components=1)
X_umap_lda = lda_on_umap.fit_transform(X_umap, y_umap)

In [ ]:
scores = X_umap_lda.ravel()  


lda_umap_auc = plot_roc_auc(scores, y_umap,  path=str(output_dir / "umap_roc_auc_for_lda.png"))

In [ ]:
plot_kde(X_umap_lda, y_umap, path=str(output_dir / "umap_kde_for_lda.png"))

#### 4.3 Комбинированный:

In [ ]:
X_comb = np.vstack((test_cifar9_CR_pu, test_cifar9_WR_pu, train_target_pu))

# 0 = старые классы (CR+WR), 1 = новый unseen класс
y_comb = np.hstack((
    np.zeros(len(test_cifar9_CR_pu) + len(test_cifar9_WR_pu)),
    np.ones(len(train_target_pu))
))

lda_on_comb = LinearDiscriminantAnalysis(n_components=1)
X_comb_lda = lda_on_comb.fit_transform(X_comb, y_comb)

In [ ]:
scores = X_comb_lda.ravel()  

lda_comb_auc = plot_roc_auc(scores, y_comb, path=str(output_dir / "comb_roc_auc_for_lda.png"))

In [ ]:
plot_kde(X_comb_lda, y_comb, path=str(output_dir / "comb_kde_for_lda.png"))

###  5. Новый пайплайн

In [ ]:
def new_pipeline(loader: DataLoader, thresh: int, lda, proj):
    features, preds, gt_preds = extract_features(resnet.model, loader, device)
    features_p = proj.transform(features)


    lda_predict =  np.where(lda.predict_proba(features_p)[:, 1] > thresh, 1, 0)

    print(preds.shape, lda_predict.shape)

    predict = []
    for i in range(lda_predict.shape[0]):
        if lda_predict[i]:
            predict.append(target)
        else:
            predict.append(preds[i])

    return np.array(predict), gt_preds


In [ ]:
def print_metric(preds, gt_preds):
    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
    new_pipeline_loader_f1_score_macro = f1_score(gt_preds, preds, average="macro")
    new_pipeline_loader_f1_score_micro = f1_score(gt_preds, preds, average="micro")
    new_pipeline_loader_f1_score_list = f1_score(gt_preds, preds, average=None)
    new_pipeline_loader_acc = accuracy_score(gt_preds, preds)

    print("f1-мера (глобальные TP и FP): {} \nf1-score (усреднее): {}" \
    "\nf1-score (по каждому классу): {} \nacc: {}".format(
        new_pipeline_loader_f1_score_macro,
        new_pipeline_loader_f1_score_micro,
        new_pipeline_loader_f1_score_list,
        new_pipeline_loader_acc
        )
    )

    return new_pipeline_loader_f1_score_macro, new_pipeline_loader_f1_score_list

#### 5.1 Для PCA:

In [ ]:
preds_pca, gt_preds_pca = new_pipeline(new_pipeline_loader, 0.8, lda_on_pca, pca)
f1_macro_pca, f1_list_pca = print_metric(preds_pca, gt_preds_pca)

#### 5.2 Для UMAP:

In [ ]:
preds_umap, gt_preds_umap = new_pipeline(new_pipeline_loader, 0.8, lda_on_umap, umap)

f1_macro_umap, f1_list_umap = print_metric(preds_umap, gt_preds_umap)

#### 5.3 Для COMB

In [ ]:
preds_comb, gt_preds_comb = new_pipeline(new_pipeline_loader, 0.8, lda_on_comb, umap_comb)

f1_macro_comb, f1_list_comb = print_metric(preds_comb, gt_preds_comb)

### 6. Это на посмотреть про модели resnet

In [ ]:
import pandas as pd

# df = pd.read_csv(str(OUTPUT_DIR / "output.csv"))
# df

In [ ]:
# filename = "./output.csv"
# with open(filename, mode='a', newline='') as file:
#     writer = csv.writer(file)
    
#     writer.writerow([
#         target, 
#         test_cifar9_f1_score,
#         lda_pca_auc,
#         lda_umap_auc,
#         lda_comb_auc,
#         f1_macro_pca, 
#         f1_macro_umap, 
#         f1_macro_comb,
#         test_cifar9_f1_score_list,
#         f1_list_pca,
#         f1_list_umap,
#         f1_list_comb,
#     ])

In [ ]:


df = pd.read_csv("output.csv")
df